# 第一阶段：Olist 订单数据读取与清洗

本阶段完成：

1. 读取原始文件 `data/raw/olist_orders_dataset.csv`；
2. 将 5 个时间字段统一转换为 `datetime`；
3. 检查并处理缺失值（保留业务上合理的空值）；
4. 输出清洗结果到 `data/processed/orders_cleaned.csv`，并写入 SQLite 数据库 `data/olist.db` 的 `orders_cleaned` 表。

In [1]:
# ===== 1. 导入所需库 =====
import os

import pandas as pd
from sqlalchemy import create_engine

# ===== 2. 定义路径 =====
RAW_PATH = "data/raw/olist_orders_dataset.csv"        # 原始订单数据
PROCESSED_PATH = "data/processed/orders_cleaned.csv"  # 清洗后输出
DB_PATH = "data/olist.db"                             # SQLite 数据库文件

# 确保输出目录存在（重复执行不会报错）
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

In [2]:
# ===== 3. 读取原始数据 =====
df = pd.read_csv(RAW_PATH)
print(f"原始数据规模：{df.shape[0]} 行 × {df.shape[1]} 列")
print(f"order_id 是否存在重复：{df['order_id'].duplicated().any()}")

# 需要转成 datetime 的 5 个时间字段
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# 统一转换为 pandas 的 datetime 类型，无法解析的值会变成 NaT
df[date_cols] = df[date_cols].apply(pd.to_datetime)

# ===== 缺失值检查 =====
missing = df.isna().sum()
print("\n处理前缺失值统计：")
print(missing[missing > 0])


原始数据规模：99441 行 × 8 列
order_id 是否存在重复：False

处理前缺失值统计：
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


In [3]:
# ===== 缺失值处理（结合电商订单业务特点）=====
# 1) 订单主键、客户 ID、状态、下单时间缺失时，订单无法归属或无法判断状态，直接删除；
#    本数据集这几个字段实际无缺失。
# 2) 审批/承运/签收时间缺失属于业务上的“尚未发生”（如已取消、未发货、未签收的订单），
#    不能用中位数、前向填充等方式补齐，否则会凭空造出“已审批/已发货/已签收”的时间线，
#    导致后续履约时长、延迟率等指标失真。因此保留 NaT（空值），
#    后续分析时按 order_status 过滤或单独统计即可。
key_cols = ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]
df = df.dropna(subset=key_cols).reset_index(drop=True)

# 仅统计时间字段缺失率，便于后续判断有效样本量（不做填充）
missing_rate = (df[date_cols].isna().mean() * 100).round(2)
print("各时间字段缺失率（%，保留空值）：")
print(missing_rate)


各时间字段缺失率（%，保留空值）：
order_purchase_timestamp         0.00
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
dtype: float64


In [4]:
# ===== 4. 保存清洗后的数据 =====
# 输出 CSV：utf-8-sig 便于 Excel 直接打开且不乱码
df.to_csv(PROCESSED_PATH, index=False, encoding="utf-8-sig")
print(f"已保存清洗后 CSV：{PROCESSED_PATH}")

# 创建 SQLite 引擎；if_exists="replace" 表示表已存在时先删除再重建
engine = create_engine(f"sqlite:///{DB_PATH}")
df.to_sql("orders_cleaned", engine, if_exists="replace", index=False)
print(f"已写入 SQLite：{DB_PATH} -> orders_cleaned")

# 简单校验数据库中的行数
db_rows = pd.read_sql("SELECT COUNT(*) AS cnt FROM orders_cleaned", engine).loc[0, "cnt"]
print(f"数据库表行数：{db_rows}（与 DataFrame 一致：{db_rows == len(df)}）")


已保存清洗后 CSV：data/processed/orders_cleaned.csv


已写入 SQLite：data/olist.db -> orders_cleaned
数据库表行数：99441（与 DataFrame 一致：True）


In [5]:
# ===== 5. 查看前 5 行与数据基本信息 =====
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [7]:
# ===== 1. 统一配置：原始文件、数据库表名、需要转 datetime 的时间字段 =====
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
DB_PATH = "data/olist.db"          # 与第一阶段使用同一个数据库文件

TABLES = [
    {
        "file": "olist_order_payments_dataset.csv",
        "table": "payments",
        "datetime_cols": [],                       # 支付表无需转换时间字段
    },
    {
        "file": "olist_order_items_dataset.csv",
        "table": "order_items",
        "datetime_cols": ["shipping_limit_date"],  # 卖家发货截止时间
    },
    {
        "file": "olist_order_reviews_dataset.csv",
        "table": "reviews",
        "datetime_cols": ["review_creation_date", "review_answer_timestamp"],
    },
    {
        "file": "olist_products_dataset.csv",
        "table": "products",
        "datetime_cols": [],                       # 商品表均为数值与文本字段
    },
    {
        "file": "product_category_name_translation.csv",
        "table": "category_translation",
        "datetime_cols": [],                       # 仅为品类名称映射表
    },
]


def clean_and_load(cfg, engine):
    """读取单张原始表 -> 时间字段转 datetime -> 去重 -> 存 CSV -> 写入 SQLite。"""
    raw_path = os.path.join(RAW_DIR, cfg["file"])
    out_path = os.path.join(PROCESSED_DIR, f"{cfg['table']}_cleaned.csv")

    df = pd.read_csv(raw_path)
    raw_rows = len(df)

    # 时间字段转换：errors="coerce" 让无法解析的值变成 NaT，避免整列报错中断流程
    for col in cfg["datetime_cols"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # 剔除完全重复的行（所有字段取值都相同才算重复，不会误删同一订单的多条支付/明细）
    df = df.drop_duplicates().reset_index(drop=True)

    # 输出 CSV：utf-8-sig 便于 Excel 直接打开不乱码
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    # 写入 SQLite，表已存在时先删除再重建
    df.to_sql(cfg["table"], engine, if_exists="replace", index=False)

    print(
        f"表 {cfg['table']} 已成功入库：{len(df):,} 行 × {df.shape[1]} 列"
        f"（原始 {raw_rows:,} 行，去重 {raw_rows - len(df):,} 行），CSV -> {out_path}"
    )
    return df


In [8]:
# ===== 2. 创建数据库连接，循环处理 5 张表 =====
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

engine = create_engine(f"sqlite:///{DB_PATH}")

cleaned_tables = {}
for cfg in TABLES:
    cleaned_tables[cfg["table"]] = clean_and_load(cfg, engine)

print("\n全部 5 张表处理完成。")


表 payments 已成功入库：103,886 行 × 5 列（原始 103,886 行，去重 0 行），CSV -> data/processed\payments_cleaned.csv


表 order_items 已成功入库：112,650 行 × 7 列（原始 112,650 行，去重 0 行），CSV -> data/processed\order_items_cleaned.csv


表 reviews 已成功入库：99,224 行 × 7 列（原始 99,224 行，去重 0 行），CSV -> data/processed\reviews_cleaned.csv


表 products 已成功入库：32,951 行 × 9 列（原始 32,951 行，去重 0 行），CSV -> data/processed\products_cleaned.csv
表 category_translation 已成功入库：71 行 × 2 列（原始 71 行，去重 0 行），CSV -> data/processed\category_translation_cleaned.csv

全部 5 张表处理完成。


In [9]:
# ===== 3. 入库校验：核对每张表的行数、字段与时间字段样例 =====
for cfg in TABLES:
    table = cfg["table"]
    cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table}", engine).loc[0, "cnt"]
    cols = pd.read_sql(f"SELECT * FROM {table} LIMIT 0", engine).columns.tolist()
    print(f"{table:<20} 行数={cnt:>7,}  字段={cols}")
    if cfg["datetime_cols"]:
        sample = pd.read_sql(
            f"SELECT {', '.join(cfg['datetime_cols'])} FROM {table} LIMIT 2", engine
        )
        print(sample.to_string(index=False))
    print()


payments             行数=103,886  字段=['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

order_items          行数=112,650  字段=['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
       shipping_limit_date
2017-09-19 09:45:35.000000
2017-05-03 11:05:13.000000

reviews              行数= 99,224  字段=['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
      review_creation_date    review_answer_timestamp
2018-01-18 00:00:00.000000 2018-01-18 21:46:59.000000
2018-03-10 00:00:00.000000 2018-03-11 03:05:13.000000

products             行数= 32,951  字段=['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

category_translation 行数=     71  字段=['product_category_name', 'product_